# 📦 Notebook 1: Data Preprocessing
**Project**: Agentic AI-Based Dynamic Tariff Optimization for EV Charging Networks  
**Team**: Socbiz Analytics | OP'26  

---

## Overview
This notebook loads, cleans, and engineers features from two real-world EV charging datasets:
1. **ACN-Data** – Caltech Adaptive Charging Network (30,000+ sessions)
2. **UrbanEV ST-EVCDP** – Shenzhen city-wide charging grid (24,798 piles, 5-min intervals)

### Engineered Features
| Feature | Formula |
|---------|---------|
| `charger_utilization_rate` | Charging Time / Total Available Time |
| `revenue_per_session` | kWh × ₹15 base tariff |
| `energy_cost_per_kwh` | price_multiplier × ₹8 wholesale cost |
| `queue_length_proxy` | max(0, occupancy − total_piles) |
| `occupancy_density` | active_sessions / total_slots |
| `is_peak` | 1 if hour ∈ {8–10 h, 17–20 h} |

### Assumptions (documented)
- Missing occupancy → filled with 0 (station offline / no data reported)
- ACN concurrent capacity: 54 slots (Caltech facility design spec)
- ST-EVCDP kWh proxy: occupancy × 7.2 kW × (5/60 hr) per 5-min interval


In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

BASE_DIR  = "/Users/saksham56/Desktop/open-projects/Socbiz Analytics Project"
ACN_PATH  = os.path.join(BASE_DIR, "ACN Data_ 25 April 2018 to 16 Dec 2018",
                         "acndata_sessions.json.xlsx")
STEVC_DIR = os.path.join(BASE_DIR, "UrbanEV_ SZ_districts")
PROC_DIR  = os.path.join(BASE_DIR, "data", "processed")
os.makedirs(PROC_DIR, exist_ok=True)

print("=" * 60)
print("  EV Charging Data Preprocessing Pipeline")
print("=" * 60)
print(f"  ACN file : {os.path.basename(ACN_PATH)}")
print(f"  ACN exists: {os.path.exists(ACN_PATH)}")
print(f"  STEVC dir : {STEVC_DIR}")
print(f"  Output    : {PROC_DIR}")


  EV Charging Data Preprocessing Pipeline
  ACN file : acndata_sessions.json.xlsx
  ACN exists: True
  STEVC dir : /Users/saksham56/Desktop/open-projects/Socbiz Analytics Project/UrbanEV_ SZ_districts
  Output    : /Users/saksham56/Desktop/open-projects/Socbiz Analytics Project/data/processed


## Step 1 – ACN-Data Preprocessing

In [2]:
print("\n[1/3] Loading ACN-Data ...")
df_acn = pd.read_excel(ACN_PATH)
print(f"  Raw shape  : {df_acn.shape}")
print(f"  Columns    : {list(df_acn.columns)}")



[1/3] Loading ACN-Data ...


  Raw shape  : (16304, 27)
  Columns    : ['_meta', 'end', 'min_kWh', 'site', 'start', '_items', '_id', 'clusterID', 'connectionTime', 'disconnectTime', 'doneChargingTime', 'kWhDelivered', 'sessionID', 'siteID', 'spaceID', 'stationID', 'timezone', 'userID', 'userInputs', 'WhPerMile', 'kWhRequested', 'milesRequested', 'minutesAvailable', 'modifiedAt', 'paymentRequired', 'requestedDeparture', 'userID.1']


In [3]:
# Parse timestamps (ACN uses ISO8601 with timezone)
for col in ['connectionTime', 'disconnectTime', 'doneChargingTime']:
    if col in df_acn.columns:
        df_acn[col] = (pd.to_datetime(df_acn[col], utc=True, errors='coerce')
                         .dt.tz_convert('America/Los_Angeles')
                         .dt.tz_localize(None))

df_acn = df_acn.dropna(subset=['connectionTime', 'disconnectTime', 'kWhDelivered'])

# ── Feature Engineering ──────────────────────────────────────
df_acn['session_duration_hr']  = (
    (df_acn['disconnectTime'] - df_acn['connectionTime']).dt.total_seconds() / 3600)
df_acn['charging_duration_hr'] = (
    (df_acn['doneChargingTime'] - df_acn['connectionTime']).dt.total_seconds() / 3600)
df_acn['idle_time_hr'] = (
    df_acn['session_duration_hr'] - df_acn['charging_duration_hr']).clip(lower=0)

# Charger utilisation rate = charging_hr / session_hr  (clamped 0-1)
df_acn['charger_utilization_rate'] = (
    df_acn['charging_duration_hr'] /
    df_acn['session_duration_hr'].replace(0, np.nan)
).clip(0, 1)

# Time features
df_acn['hour']        = df_acn['connectionTime'].dt.hour
df_acn['day_of_week'] = df_acn['connectionTime'].dt.dayofweek
df_acn['is_weekend']  = (df_acn['day_of_week'] >= 5).astype(int)
df_acn['is_peak']     = (
    df_acn['hour'].between(8, 10) | df_acn['hour'].between(17, 20)).astype(int)

# Revenue & cost proxies
BASE_TARIFF = 15.0
df_acn['energy_cost_per_kwh'] = 8.0
df_acn['revenue_per_session'] = df_acn['kWhDelivered'] * BASE_TARIFF

# Occupancy density — ASSUMPTION: 54 concurrent slots max per site
df_acn['hour_date'] = df_acn['connectionTime'].dt.floor('h')
active = (df_acn.groupby(['siteID','hour_date'])['sessionID']
          .count().reset_index(name='active_sessions'))
df_acn = df_acn.merge(active, on=['siteID','hour_date'], how='left')
df_acn['occupancy_density'] = df_acn['active_sessions'] / 54

print(f"  Cleaned shape : {df_acn.shape}")
print(f"  Date range    : {df_acn['connectionTime'].min().date()} → {df_acn['connectionTime'].max().date()}")
print(f"  Sites         : {sorted(df_acn['siteID'].unique())}")
print(f"  Mean session duration  : {df_acn['session_duration_hr'].mean():.2f} h")
print(f"  Mean idle time         : {df_acn['idle_time_hr'].mean():.2f} h")
print(f"  Mean kWh delivered     : {df_acn['kWhDelivered'].mean():.2f}")
print(f"  Mean util rate         : {df_acn['charger_utilization_rate'].mean():.2%}")


  Cleaned shape : (14999, 40)
  Date range    : 2018-04-25 → 2018-12-15
  Sites         : [np.float64(2.0)]
  Mean session duration  : 5.92 h
  Mean idle time         : 2.69 h
  Mean kWh delivered     : 9.00
  Mean util rate         : 69.15%


In [4]:
acn_out = os.path.join(PROC_DIR, "acn_clean.csv")
df_acn.to_csv(acn_out, index=False)
print(f"  ✓ ACN saved → {acn_out}  shape={df_acn.shape}")


  ✓ ACN saved → /Users/saksham56/Desktop/open-projects/Socbiz Analytics Project/data/processed/acn_clean.csv  shape=(14999, 40)


## Step 2 – ST-EVCDP (UrbanEV Shenzhen) Preprocessing

The ST-EVCDP dataset is stored in **wide matrix format** (rows = timestamps, columns = grid IDs).  
Files: `time.csv` (timestamps), `occupancy.csv` (vehicles per grid), `price.csv` (price multiplier), `volume.csv` (charging volume), `information.csv` (grid metadata).


In [5]:
print("\n[2/3] Processing ST-EVCDP files ...")

# ── Load component files ─────────────────────────────────────
time_df  = pd.read_csv(os.path.join(STEVC_DIR, "time.csv"))
occ_df   = pd.read_csv(os.path.join(STEVC_DIR, "occupancy.csv"))
price_df = pd.read_csv(os.path.join(STEVC_DIR, "price.csv"))
vol_df   = pd.read_csv(os.path.join(STEVC_DIR, "volume.csv"))
info_df  = pd.read_csv(os.path.join(STEVC_DIR, "information.csv"))

print(f"  time.csv      : {time_df.shape}")
print(f"  occupancy.csv : {occ_df.shape}  (rows=timestamps, cols=grids)")
print(f"  price.csv     : {price_df.shape}")
print(f"  volume.csv    : {vol_df.shape}")
print(f"  information   : {info_df.shape}")

# Reconstruct datetime from components
time_df['datetime'] = pd.to_datetime({
    'year'  : time_df['year'],
    'month' : time_df['month'],
    'day'   : time_df['day'],
    'hour'  : time_df['hour'],
    'minute': time_df['minute'],
})
print(f"  Date range: {time_df['datetime'].min()} → {time_df['datetime'].max()}")



[2/3] Processing ST-EVCDP files ...


  time.csv      : (8640, 6)
  occupancy.csv : (8640, 248)  (rows=timestamps, cols=grids)
  price.csv     : (8640, 248)
  volume.csv    : (8640, 248)
  information   : (247, 10)
  Date range: 2022-06-19 00:00:00 → 2022-07-18 23:55:00


In [6]:
# ── Melt wide→long format ────────────────────────────────────
grid_cols = [c for c in occ_df.columns if c != 'timestamp']

# Occupancy
occ_long = occ_df.melt(id_vars=['timestamp'], value_vars=grid_cols,
                        var_name='grid_id', value_name='occupancy')
# Price
price_long = price_df.melt(id_vars=['timestamp'], value_vars=grid_cols,
                            var_name='grid_id', value_name='price_multiplier')
# Volume
vol_long = vol_df.melt(id_vars=['timestamp'], value_vars=grid_cols,
                        var_name='grid_id', value_name='volume')

# Merge all long tables
df_panel = occ_long.merge(price_long, on=['timestamp','grid_id'], how='left')
df_panel = df_panel.merge(vol_long,   on=['timestamp','grid_id'], how='left')

# Attach datetime from time.csv using timestamp index (1-based)
time_map = dict(zip(range(1, len(time_df)+1), time_df['datetime']))
df_panel['datetime'] = df_panel['timestamp'].map(time_map)
df_panel['grid_id']  = df_panel['grid_id'].astype(int)

# Merge grid metadata
df_panel = df_panel.merge(
    info_df[['num','grid','count','fast_count','slow_count','area','lon','la','CBD','dynamic_pricing']],
    left_on='grid_id', right_on='num', how='left'
)

print(f"  Panel shape (long): {df_panel.shape}")
print(f"  Grid IDs: {df_panel['grid_id'].nunique()} unique grids")


  Panel shape (long): (2134080, 16)
  Grid IDs: 247 unique grids


In [7]:
# ── Feature Engineering ─────────────────────────────────────
# Total piles per grid (fast + slow)
df_panel['total_piles'] = df_panel['fast_count'].fillna(0) + df_panel['slow_count'].fillna(0)
df_panel['total_piles'] = df_panel['total_piles'].replace(0, np.nan)

# Charger utilisation rate = occupancy / total_piles  (clamped 0–1)
df_panel['charger_utilization_rate'] = (
    df_panel['occupancy'] / df_panel['total_piles']).clip(0, 1)

# kWh proxy = occupancy × 7.2 kW × (5/60) hr per 5-min interval
df_panel['energy_kwh'] = df_panel['occupancy'] * 7.2 * (5 / 60)

# Time features
df_panel['hour']        = df_panel['datetime'].dt.hour
df_panel['day_of_week'] = df_panel['datetime'].dt.dayofweek
df_panel['is_weekend']  = (df_panel['day_of_week'] >= 5).astype(int)
df_panel['is_peak']     = (
    df_panel['hour'].between(8, 10) | df_panel['hour'].between(17, 20)).astype(int)

df_panel['occupancy_density']   = df_panel['charger_utilization_rate']
df_panel['queue_length_proxy']  = (df_panel['occupancy'] - df_panel['total_piles']).clip(lower=0)
df_panel['energy_cost_per_kwh'] = df_panel['price_multiplier'] * 8.0

# Missing value imputation — ASSUMPTION: NaN occupancy = zero activity
for col in ['occupancy','charger_utilization_rate','energy_kwh',
            'price_multiplier','CBD','dynamic_pricing','occupancy_density',
            'queue_length_proxy','volume']:
    if col in df_panel.columns:
        df_panel[col] = df_panel[col].fillna(0 if col != 'price_multiplier' else 1.0)

print(f"  Final panel shape: {df_panel.shape}")
print(f"  Date range: {df_panel['datetime'].min()} → {df_panel['datetime'].max()}")
print(f"  Unique grids: {df_panel['grid_id'].nunique()}")
print(f"  Mean charger utilisation: {df_panel['charger_utilization_rate'].mean():.2%}")
print(f"  Mean occupancy: {df_panel['occupancy'].mean():.2f} vehicles")


  Final panel shape: (2134080, 26)
  Date range: 2022-06-19 00:00:00 → 2022-07-18 23:55:00
  Unique grids: 247
  Mean charger utilisation: 7.23%
  Mean occupancy: 21.90 vehicles


In [8]:
panel_out = os.path.join(PROC_DIR, "stevcdp_panel.csv")
df_panel.to_csv(panel_out, index=False)
print(f"  ✓ ST-EVCDP panel saved → {panel_out}  shape={df_panel.shape}")


  ✓ ST-EVCDP panel saved → /Users/saksham56/Desktop/open-projects/Socbiz Analytics Project/data/processed/stevcdp_panel.csv  shape=(2134080, 26)


## Step 3 – Unified Features Dataset

In [9]:
print("\n[3/3] Building unified features dataset ...")

# ACN → hourly aggregates per site
df_acn['siteID_num'] = df_acn['siteID'].astype('category').cat.codes + 1
df_acn['hour_dt']    = df_acn['connectionTime'].dt.floor('h')

acn_h = df_acn.groupby(['hour_dt','siteID_num']).agg(
    utilization_rate  = ('charger_utilization_rate', 'mean'),
    energy_kwh        = ('kWhDelivered', 'sum'),
    occupancy_density = ('occupancy_density', 'mean'),
    session_count     = ('sessionID', 'count')
).reset_index().rename(columns={'hour_dt':'datetime_hour','siteID_num':'location_id'})
acn_h['dataset_source'] = 'ACN'

# ST-EVCDP → hourly aggregates per grid
df_panel['datetime_hour'] = df_panel['datetime'].dt.floor('h')
stevc_h = df_panel.groupby(['datetime_hour','grid_id']).agg(
    utilization_rate  = ('charger_utilization_rate','mean'),
    energy_kwh        = ('energy_kwh','sum'),
    occupancy_density = ('occupancy_density','mean')
).reset_index().rename(columns={'grid_id':'location_id'})
stevc_h['session_count']  = np.nan
stevc_h['dataset_source'] = 'ST_EVCDP'

# Concatenate
cols = ['datetime_hour','location_id','utilization_rate','energy_kwh',
        'occupancy_density','session_count','dataset_source']
unified = pd.concat([acn_h[cols], stevc_h[cols]], ignore_index=True)
unified['hour']        = unified['datetime_hour'].dt.hour
unified['day_of_week'] = unified['datetime_hour'].dt.dayofweek
unified['is_weekend']  = (unified['day_of_week'] >= 5).astype(int)
unified['is_peak']     = (unified['hour'].between(8,10) | unified['hour'].between(17,20)).astype(int)

u_out = os.path.join(PROC_DIR, "unified_features.csv")
unified.to_csv(u_out, index=False)
print(f"  ✓ Unified saved → {u_out}  shape={unified.shape}")
print(f"  Source breakdown:\n{unified['dataset_source'].value_counts().to_string()}")
print(f"  Date range: {unified['datetime_hour'].min()} → {unified['datetime_hour'].max()}")



[3/3] Building unified features dataset ...


  ✓ Unified saved → /Users/saksham56/Desktop/open-projects/Socbiz Analytics Project/data/processed/unified_features.csv  shape=(181832, 11)
  Source breakdown:
dataset_source
ST_EVCDP    177840
ACN           3992
  Date range: 2018-04-25 04:00:00 → 2022-07-18 23:00:00


In [10]:
# ── FINAL SUMMARY ───────────────────────────────────────────
print("\n" + "="*60)
print("  PREPROCESSING SUMMARY")
print("="*60)
print(f"  ACN Sessions           : {len(df_acn):,}")
print(f"  ACN Sites              : {df_acn['siteID'].nunique()}")
print(f"  ACN Date Range         : {df_acn['connectionTime'].min().date()} → "
      f"{df_acn['connectionTime'].max().date()}")
print(f"  ST-EVCDP Intervals     : {len(df_panel):,}")
print(f"  ST-EVCDP Grid Zones    : {df_panel['grid_id'].nunique()}")
print(f"  ST-EVCDP Date Range    : {df_panel['datetime'].min().date()} → "
      f"{df_panel['datetime'].max().date()}")
print(f"  Unified Hourly Rows    : {len(unified):,}")
print("="*60)
print("  Features Engineered:")
print("    charger_utilization_rate, revenue_per_session,")
print("    energy_cost_per_kwh, queue_length_proxy,")
print("    occupancy_density, is_peak, is_weekend")
print("="*60)
print("  ✓ All files saved to data/processed/")



  PREPROCESSING SUMMARY
  ACN Sessions           : 14,999
  ACN Sites              : 1
  ACN Date Range         : 2018-04-25 → 2018-12-15
  ST-EVCDP Intervals     : 2,134,080
  ST-EVCDP Grid Zones    : 247
  ST-EVCDP Date Range    : 2022-06-19 → 2022-07-18
  Unified Hourly Rows    : 181,832
  Features Engineered:
    charger_utilization_rate, revenue_per_session,
    energy_cost_per_kwh, queue_length_proxy,
    occupancy_density, is_peak, is_weekend
  ✓ All files saved to data/processed/
